# BCMID Kaggle GPU Training

Runs the Phase 1 single-modality baseline on Kaggle and writes outputs to `/kaggle/working/results/`.

In [ ]:
from pathlib import Path
import shutil
import subprocess
import sys
import zipfile

import torch

PREFERRED_DATA_ROOT = Path('/kaggle/input/datasets/cs24m1005sindhiyar/bcmid-dataset/BCMID')
WORK_ROOT = Path('/kaggle/working')
PROJECT_ROOT = WORK_ROOT / 'BCMID_Project'
RESULTS_ROOT = WORK_ROOT / 'results'

def resolve_data_root():
    if PREFERRED_DATA_ROOT.exists():
        return PREFERRED_DATA_ROOT
    matches = sorted(Path('/kaggle/input').rglob('BCMID_labels.csv'))
    if not matches:
        raise FileNotFoundError('Could not find BCMID_labels.csv under /kaggle/input')
    return matches[0].parent

DATA_ROOT = resolve_data_root()
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)

print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
print('Dataset root:', DATA_ROOT)
print('Dataset root exists:', DATA_ROOT.exists())

In [ ]:
def find_extracted_project_root(base: Path):
    candidates = sorted(base.rglob('code/train_single.py'))
    if not candidates:
        return None
    return candidates[0].parents[1]

def ensure_project_code():
    train_script = PROJECT_ROOT / 'code' / 'train_single.py'
    if train_script.exists():
        print('Using existing project code:', PROJECT_ROOT)
        return

    zip_files = sorted(Path('/kaggle/input').rglob('*.zip'))
    for zip_path in zip_files:
        with zipfile.ZipFile(zip_path) as zf:
            names = zf.namelist()
            if not any(name.endswith('code/train_single.py') for name in names):
                continue
            extract_root = WORK_ROOT / 'uploaded_bcmid_project'
            if extract_root.exists():
                shutil.rmtree(extract_root)
            extract_root.mkdir(parents=True, exist_ok=True)
            zf.extractall(extract_root)
            found_root = find_extracted_project_root(extract_root)
            if found_root is None:
                continue
            if PROJECT_ROOT.exists():
                shutil.rmtree(PROJECT_ROOT)
            shutil.copytree(found_root, PROJECT_ROOT)
            print('Extracted project code from:', zip_path)
            print('Project root:', PROJECT_ROOT)
            return

    raise FileNotFoundError('Could not find uploaded project ZIP containing code/train_single.py under /kaggle/input')

ensure_project_code()

In [ ]:
requirements = PROJECT_ROOT / 'requirements.txt'
if requirements.exists():
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(requirements)])
else:
    print('No requirements.txt found; using Kaggle environment packages.')

In [ ]:
cmd = [
    sys.executable,
    'code/train_single.py',
    '--modality', 'ultrasound',
    '--backbone', 'efficientnet_b0',
    '--data-root', str(DATA_ROOT),
    '--output-dir', str(RESULTS_ROOT),
    '--weighted-bce',
]
print('Running:', ' '.join(cmd))
subprocess.check_call(cmd, cwd=str(PROJECT_ROOT))

In [ ]:
print('Saved outputs:')
for path in sorted(RESULTS_ROOT.rglob('*')):
    if path.is_file():
        print(path)